In [1]:
# ============================================================
# CELL 1
# IMPORTS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import time

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


Mounted at /content/drive


In [2]:
# ============================================================
# CELL 2
# DATASET SETUP
# ============================================================

DATASET_ROOT = os.path.join(r"c:\Users\priya\Downloads\ok\TinyBayes-ML", "data", "dataset", "rice")

SAVE_DIR = os.path.join(
    DATASET_ROOT,
    "run"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

RANDOM_STATE = 42

IMAGE_SIZE = 224

CLASS_NAMES = ['Bacterial_Leaf_Blight', 'Brown_Spot', 'Healthy', 'Leaf_Blast', 'Leaf_Scald', 'Sheath_Blight']

print("="*60)
print("CLASSES")
print("="*60)

print(CLASS_NAMES)

# ------------------------------------------------------------
# Build dataframe from dataset directory
# ------------------------------------------------------------

data = []
for cls_name in CLASS_NAMES:
    cls_dir = os.path.join(DATASET_ROOT, cls_name)
    if os.path.exists(cls_dir):
        for img_file in os.listdir(cls_dir):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                data.append({'Image_ID': img_file, 'class': cls_name})

df = pd.DataFrame(data)

print()
print("Total Images :", len(df))
print()
print(df["class"].value_counts())

# ------------------------------------------------------------
# Encode labels
# ------------------------------------------------------------

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["class"])
NUM_CLASSES = len(label_encoder.classes_)

print()
print("Label Mapping")
for i, cls in enumerate(label_encoder.classes_):
    print(i, "->", cls)

# ------------------------------------------------------------
# Train / Validation Split (Stratified 80 / 20)
# ------------------------------------------------------------

df_train, df_val = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=RANDOM_STATE
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print()
print("="*60)
print("TRAIN SPLIT")
print("="*60)
print(df_train["class"].value_counts())

print()
print("="*60)
print("VALIDATION SPLIT")
print("="*60)
print(df_val["class"].value_counts())

# ------------------------------------------------------------
# Save splits
# ------------------------------------------------------------

df_train.to_csv(os.path.join(SAVE_DIR, "train_split.csv"), index=False)
df_val.to_csv(os.path.join(SAVE_DIR, "validation_split.csv"), index=False)

print()
print("Saved")
print(os.path.join(SAVE_DIR, "train_split.csv"))
print(os.path.join(SAVE_DIR, "validation_split.csv"))


CLASSES
['Bacterial_Leaf_Blight', 'Brown_Spot', 'Healthy', 'Leaf_Blast', 'Leaf_Scald', 'Sheath_Blight']

Total Images : 3829

class
Healthy                  653
Brown_Spot               646
Bacterial_Leaf_Blight    636
Leaf_Blast               634
Sheath_Blight            632
Leaf_Scald               628

Label Mapping
0 -> Bacterial_Leaf_Blight
1 -> Brown_Spot
2 -> Healthy
3 -> Leaf_Blast
4 -> Leaf_Scald
5 -> Sheath_Blight

TRAIN SPLIT
class
Healthy                  522
Brown_Spot               517
Bacterial_Leaf_Blight    509
Leaf_Blast               507
Sheath_Blight            506
Leaf_Scald               502

VALIDATION SPLIT
class
Healthy                  131
Brown_Spot               129
Bacterial_Leaf_Blight    127
Leaf_Blast               127
Leaf_Scald               126
Sheath_Blight            126

Saved
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\rice\run\train_split.csv
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\rice\run\validation_split.csv


In [3]:
train_ids = set(df_train["Image_ID"])
val_ids = set(df_val["Image_ID"])

overlap = train_ids.intersection(val_ids)

print("Overlap:", len(overlap))


Overlap: 0


In [4]:
# ============================================================
# SAVE CLASS NAMES
# ============================================================

CLASS_NAMES_PATH = os.path.join(
    SAVE_DIR,
    "class_names.json"
)

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(CLASS_NAMES, f, indent=4)

print("Class names saved to:")
print(CLASS_NAMES_PATH)

print()
print("Classes:")
print(CLASS_NAMES)


Class names saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\rice\run\class_names.json

Classes:
['Bacterial_Leaf_Blight', 'Brown_Spot', 'Healthy', 'Leaf_Blast', 'Leaf_Scald', 'Sheath_Blight']


In [5]:
# ============================================================
# CELL 3
# EXTRACT MOBILENET FEATURES
# ============================================================

TRAIN_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "train_features.csv"
)

VAL_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "validation_features.csv"
)

if os.path.exists(TRAIN_FEATURE_PATH) and os.path.exists(VAL_FEATURE_PATH):

    print("="*60)
    print("FEATURE FILES FOUND")
    print("="*60)

    train_features_df = pd.read_csv(TRAIN_FEATURE_PATH)
    val_features_df = pd.read_csv(VAL_FEATURE_PATH)

    print(train_features_df.shape)
    print(val_features_df.shape)
    print()
    print(train_features_df.head())


FEATURE FILES FOUND
(3063, 578)
(766, 578)

                  Image_ID                  class    feat_0    feat_1    feat_2
0           aug_0_8185.jpg          Sheath_Blight  0.117290  0.319762  0.396161
1            aug_0_549.jpg          Sheath_Blight  0.160366  0.072769 -0.180852
2           aug_0_2013.jpg  Bacterial_Leaf_Blight -0.051280 -0.096632  0.219237
3           aug_0_8060.jpg             Leaf_Blast  0.030827 -0.025527  0.449758
4  IMG_20231014_170949.jpg          Sheath_Blight  0.261907 -0.030447  0.161864

Train features shape: (3063, 578)
Val features shape:   (766, 578)


In [6]:
# ============================================================
# CELL 4
# TRAIN JACOBI-DMR
# ============================================================

import time

time_jacobi_start = time.time()

# ------------------------------------------------------------
# Separate features and labels
# ------------------------------------------------------------

feat_cols = [c for c in train_features_df.columns if c.startswith("feat_")]

X_train = train_features_df[feat_cols].values.astype(np.float32)
y_train = train_features_df["class"].values

X_val = val_features_df[feat_cols].values.astype(np.float32)
y_val = val_features_df["class"].values

# ------------------------------------------------------------
# One-hot encode targets
# ------------------------------------------------------------

y_train_df = pd.DataFrame({"c": y_train})
y_one_hot = pd.get_dummies(y_train_df["c"])

for cls in CLASS_NAMES:
    if cls not in y_one_hot.columns:
        y_one_hot[cls] = 0

y_one_hot = y_one_hot[CLASS_NAMES]

# ------------------------------------------------------------
# Jacobi regularized solve
# ------------------------------------------------------------

N = len(X_train)
a = 1.0 / N
b = 1.0 / N
k = 1.0

identity = np.eye(X_train.shape[1], dtype=np.float32)
regularization_lambda = 1.0

XtX = np.matmul(X_train.T, X_train)
XtX_reg = XtX + regularization_lambda * identity

jacobi_coefficients = {}

for cls in CLASS_NAMES:
    y_c = y_one_hot[cls].values
    eta = np.log((y_c + a) / (1.0 + k * b))
    Xty = np.matmul(X_train.T, eta)
    beta = np.linalg.solve(XtX_reg, Xty)
    jacobi_coefficients[cls] = beta

# ------------------------------------------------------------
# Validation inference
# ------------------------------------------------------------

preds_jacobi = []
for i in range(len(X_val)):
    x = X_val[i]
    scores = {cls: float(np.dot(x, jacobi_coefficients[cls])) for cls in CLASS_NAMES}
    preds_jacobi.append(max(scores, key=scores.get))

time_jacobi = time.time() - time_jacobi_start
acc_jacobi = accuracy_score(y_val, preds_jacobi)

print()
print("="*60)
print("JACOBI-DMR")
print("="*60)
print(f"Accuracy: {acc_jacobi:.4f}")
print(f"Time: {time_jacobi:.4f}s")
print()
print(classification_report(y_val, preds_jacobi, labels=CLASS_NAMES))
print()
print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_jacobi, labels=CLASS_NAMES))

# ------------------------------------------------------------
# Save coefficients
# ------------------------------------------------------------

JACOBI_JSON_PATH = os.path.join(SAVE_DIR, "jacobi_coefficients.json")

save_dict = {cls: [float(v) for v in jacobi_coefficients[cls]] for cls in CLASS_NAMES}

with open(JACOBI_JSON_PATH, "w") as f:
    json.dump(save_dict, f, indent=4)

print()
print("Jacobi coefficients saved to:")
print(JACOBI_JSON_PATH)



JACOBI-DMR
Accuracy: 0.8681
Time: 2.6841s

                       precision    recall  f1-score   support

Bacterial_Leaf_Blight       0.79      0.87      0.83       127
           Brown_Spot       0.92      0.79      0.85       129
              Healthy       0.91      0.95      0.93       131
           Leaf_Blast       0.82      0.84      0.83       127
           Leaf_Scald       0.87      0.83      0.85       126
        Sheath_Blight       0.91      0.92      0.92       126

             accuracy                           0.87       766
            macro avg       0.87      0.87      0.87       766
         weighted avg       0.87      0.87      0.87       766


Confusion Matrix:

[[111   2   0  10   4   0]
 [  5 102  10   6   3   3]
 [  0   2 125   2   1   1]
 [  8   3   1 107   4   4]
 [ 13   1   1   4 104   3]
 [  4   1   0   2   3 116]]

Jacobi coefficients saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\rice\run\jacobi_coefficients.json


In [7]:
# ============================================================
# RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier

import time

time_rf_start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

preds_rf = rf_model.predict(X_val)

time_rf = time.time() - time_rf_start

acc_rf = accuracy_score(y_val, preds_rf)

print()
print("="*60)
print("RANDOM FOREST")
print("="*60)

print(f"Accuracy: {acc_rf:.4f}")
print(f"Time: {time_rf:.4f}s")
print()

print(classification_report(y_val, preds_rf, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_rf, labels=CLASS_NAMES))



RANDOM FOREST
Accuracy: 0.8433
Time: 12.7352s

                       precision    recall  f1-score   support

Bacterial_Leaf_Blight       0.75      0.84      0.80       127
           Brown_Spot       0.88      0.81      0.84       129
              Healthy       0.91      0.97      0.94       131
           Leaf_Blast       0.89      0.80      0.84       127
           Leaf_Scald       0.80      0.77      0.78       126
        Sheath_Blight       0.84      0.87      0.85       126

             accuracy                           0.84       766
            macro avg       0.84      0.84      0.84       766
         weighted avg       0.85      0.84      0.84       766


Confusion Matrix:

[[107   1   1   8   6   4]
 [  3 105   3   1  11   6]
 [  0   3 127   0   1   0]
 [ 11   3   1 101   5   6]
 [ 16   4   2   2  97   5]
 [  5   4   5   1   2 109]]


In [8]:
# ============================================================
# SUPPORT VECTOR MACHINE (SVM)
# ============================================================

from sklearn.svm import SVC

import time

time_svm_start = time.time()

svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    random_state=42
)

svm_model.fit(X_train, y_train)

preds_svm = svm_model.predict(X_val)

time_svm = time.time() - time_svm_start

acc_svm = accuracy_score(y_val, preds_svm)

print()
print("="*60)
print("SUPPORT VECTOR MACHINE")
print("="*60)

print(f"Accuracy: {acc_svm:.4f}")
print(f"Time: {time_svm:.4f}s")
print()

print(classification_report(y_val, preds_svm, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_svm, labels=CLASS_NAMES))



SUPPORT VECTOR MACHINE
Accuracy: 0.8773
Time: 7.6764s

                       precision    recall  f1-score   support

Bacterial_Leaf_Blight       0.78      0.84      0.81       127
           Brown_Spot       0.93      0.86      0.90       129
              Healthy       0.95      0.98      0.97       131
           Leaf_Blast       0.85      0.82      0.84       127
           Leaf_Scald       0.86      0.85      0.86       126
        Sheath_Blight       0.89      0.90      0.90       126

             accuracy                           0.88       766
            macro avg       0.88      0.88      0.88       766
         weighted avg       0.88      0.88      0.88       766


Confusion Matrix:

[[107   0   0  12   5   3]
 [  4 111   6   2   3   3]
 [  0   1 129   0   0   1]
 [  8   2   0 104   6   7]
 [ 13   3   0   3 107   0]
 [  5   2   1   1   3 114]]


In [9]:
# ============================================================
# RIDGE CLASSIFIER
# ============================================================

from sklearn.linear_model import RidgeClassifier

import time

time_ridge_start = time.time()

ridge_model = RidgeClassifier(
    alpha=1.0,
    random_state=42
)

ridge_model.fit(X_train, y_train)

preds_ridge = ridge_model.predict(X_val)

time_ridge = time.time() - time_ridge_start

acc_ridge = accuracy_score(y_val, preds_ridge)

print()
print("="*60)
print("RIDGE CLASSIFIER")
print("="*60)

print(f"Accuracy: {acc_ridge:.4f}")
print(f"Time: {time_ridge:.4f}s")
print()

print(classification_report(y_val, preds_ridge, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_ridge, labels=CLASS_NAMES))



RIDGE CLASSIFIER
Accuracy: 0.8695
Time: 0.1322s

                       precision    recall  f1-score   support

Bacterial_Leaf_Blight       0.80      0.87      0.83       127
           Brown_Spot       0.91      0.79      0.85       129
              Healthy       0.91      0.95      0.93       131
           Leaf_Blast       0.81      0.84      0.83       127
           Leaf_Scald       0.88      0.83      0.85       126
        Sheath_Blight       0.91      0.93      0.92       126

             accuracy                           0.87       766
            macro avg       0.87      0.87      0.87       766
         weighted avg       0.87      0.87      0.87       766


Confusion Matrix:

[[110   3   0  10   4   0]
 [  4 102  10   7   3   3]
 [  0   2 125   2   1   1]
 [  8   3   1 107   4   4]
 [ 12   1   1   4 105   3]
 [  3   1   0   2   3 117]]


In [10]:
# ============================================================
# LOGISTIC REGRESSION (L1 / LASSO)
# ============================================================

from sklearn.linear_model import LogisticRegression

import time

time_lr_start = time.time()

lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

preds_lr = lr_model.predict(X_val)

time_lr = time.time() - time_lr_start

acc_lr = accuracy_score(y_val, preds_lr)

print()
print("="*60)
print("LOGISTIC REGRESSION (L1 / LASSO)")
print("="*60)

print(f"Accuracy: {acc_lr:.4f}")
print(f"Time: {time_lr:.4f}s")
print()

print(classification_report(y_val, preds_lr, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_lr, labels=CLASS_NAMES))



LOGISTIC REGRESSION (L1 / LASSO)
Accuracy: 0.8890
Time: 26.5056s

                       precision    recall  f1-score   support

Bacterial_Leaf_Blight       0.84      0.89      0.87       127
           Brown_Spot       0.96      0.82      0.89       129
              Healthy       0.96      0.98      0.97       131
           Leaf_Blast       0.83      0.87      0.85       127
           Leaf_Scald       0.85      0.86      0.85       126
        Sheath_Blight       0.90      0.91      0.91       126

             accuracy                           0.89       766
            macro avg       0.89      0.89      0.89       766
         weighted avg       0.89      0.89      0.89       766


Confusion Matrix:

[[113   0   0   9   5   0]
 [  3 106   4   6   5   5]
 [  0   1 129   0   1   0]
 [  7   1   0 110   4   5]
 [  7   2   0   6 108   3]
 [  4   0   1   2   4 115]]


In [11]:
# ============================================================
# FINAL MODEL COMPARISON
# ============================================================

comparison_df = pd.DataFrame({
    "Model": [
        "Support Vector Machine",
        "Ridge Classifier",
        "Logistic Regression (L1)",
        "Jacobi-DMR",
        "Random Forest"
    ],
    "Accuracy": [
        acc_svm,
        acc_ridge,
        acc_lr,
        acc_jacobi,
        acc_rf
    ],
    "Execution Time (s)": [
        time_svm,
        time_ridge,
        time_lr,
        time_jacobi,
        time_rf
    ]
})

comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print()
print("="*70)
print("MODEL COMPARISON")
print("="*70)

print(comparison_df.to_string(index=False))

comparison_path = os.path.join(
    SAVE_DIR,
    "model_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print()
print(f"Comparison saved to:\n{comparison_path}")



MODEL COMPARISON
                   Model  Accuracy  Execution Time (s)
Logistic Regression (L1)    0.8890             26.5056
  Support Vector Machine    0.8773              7.6764
        Ridge Classifier    0.8695              0.1322
              Jacobi-DMR    0.8681              2.6841
           Random Forest    0.8433             12.7352

Comparison saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\rice\run\model_comparison.csv
